In [1]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load dataset
df = pd.read_csv('loan_approved.csv')

In [3]:
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status (Approved)
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [4]:
df["Dependents"] = df["Dependents"].replace("3+", "3")

In [5]:
df["Dependents"] = df["Dependents"].fillna('0')
df["Dependents"] = df["Dependents"].astype(str)

In [6]:
# Drop unwanted column
df.drop('Loan_ID', axis=1, inplace=True)

In [7]:
x = df.drop('Loan_Status (Approved)', axis=1)
y = df['Loan_Status (Approved)']

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [9]:
x_train

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
83,Male,Yes,0,Graduate,No,6000,2250.0,265.0,360.0,NaN,Semiurban
90,Male,Yes,0,Graduate,No,2958,2900.0,131.0,360.0,1.0,Semiurban
227,Male,Yes,2,Graduate,No,6250,1695.0,210.0,360.0,1.0,Semiurban
482,Male,Yes,0,Graduate,No,2083,3150.0,128.0,360.0,1.0,Semiurban
464,Male,No,0,Graduate,No,4166,0.0,98.0,360.0,0.0,Semiurban
...,...,...,...,...,...,...,...,...,...,...,...
71,Male,Yes,2,Not Graduate,Yes,1875,1875.0,97.0,360.0,1.0,Semiurban
106,Male,Yes,2,Graduate,No,11417,1126.0,225.0,360.0,1.0,Urban
270,Female,No,0,Graduate,No,3237,0.0,30.0,360.0,1.0,Urban
435,Female,NaN,0,Graduate,No,10047,0.0,NaN,240.0,1.0,Semiurban


In [10]:
x_test

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
350,Male,Yes,0,Graduate,No,9083,0.0,228.0,360.0,1.0,Semiurban
377,Male,Yes,0,Graduate,No,4310,0.0,130.0,360.0,NaN,Semiurban
163,Male,Yes,2,Graduate,No,4167,1447.0,158.0,360.0,1.0,Rural
609,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural
132,Male,No,0,Graduate,No,2718,0.0,70.0,360.0,1.0,Semiurban
...,...,...,...,...,...,...,...,...,...,...,...
231,Male,Yes,0,Graduate,NaN,3716,0.0,42.0,180.0,1.0,Rural
312,Female,No,0,Graduate,No,2507,0.0,56.0,360.0,1.0,Rural
248,Male,Yes,1,Graduate,No,2882,1843.0,123.0,480.0,1.0,Semiurban
11,Male,Yes,2,Graduate,NaN,2500,1840.0,109.0,360.0,1.0,Urban


In [11]:
y_test

350    Y
377    Y
163    Y
609    Y
132    Y
      ..
231    Y
312    Y
248    Y
11     Y
333    Y
Name: Loan_Status (Approved), Length: 123, dtype: object

In [12]:
y_train

83     N
90     Y
227    Y
482    Y
464    N
      ..
71     Y
106    Y
270    Y
435    Y
102    Y
Name: Loan_Status (Approved), Length: 491, dtype: object

In [14]:
OHE_Columns = ['Gender','Married','Self_Employed','Property_Area']
Scaling_Columns = ['ApplicantIncome','CoapplicantIncome','LoanAmount']

In [15]:
def years(x):
    return x / 12

In [16]:
# Pipelines
# One Hot Encoding Pipeline
ohe_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

# Scaling Pipeline
scale_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

# Dependents Pipeline
depend_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['0', '1', '2', '3']]))])

# Education Pipeline
edu_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Not Graduate', 'Graduate']]))])

# Loan Amount Term Pipeline
loan_term_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
    ('transform', FunctionTransformer(years))])

# Credit History Pipeline
credit_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent'))])


In [17]:
# Column Transformer
# syntax: ('name', pipeline_or_transformer, column_list)
preprocessor = ColumnTransformer([('ohe', ohe_pipe, OHE_Columns),
                                  ('scaling', scale_pipe, Scaling_Columns),
                                  ('depend', depend_pipe, ['Dependents']),
                                  ('education', edu_pipe, ['Education']),
                                  ('loan_term', loan_term_pipe, ['Loan_Amount_Term']),
                                  ('credit', credit_pipe, ['Credit_History'])])

In [18]:
preprocessor

,transformers,"[('ohe', ...), ('scaling', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None


In [14]:
x_train

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
83,Male,Yes,0,Graduate,No,6000,2250.0,265.0,360.0,NaN,Semiurban
90,Male,Yes,0,Graduate,No,2958,2900.0,131.0,360.0,1.0,Semiurban
227,Male,Yes,2,Graduate,No,6250,1695.0,210.0,360.0,1.0,Semiurban
482,Male,Yes,0,Graduate,No,2083,3150.0,128.0,360.0,1.0,Semiurban
464,Male,No,0,Graduate,No,4166,0.0,98.0,360.0,0.0,Semiurban
...,...,...,...,...,...,...,...,...,...,...,...
71,Male,Yes,2,Not Graduate,Yes,1875,1875.0,97.0,360.0,1.0,Semiurban
106,Male,Yes,2,Graduate,No,11417,1126.0,225.0,360.0,1.0,Urban
270,Female,No,0,Graduate,No,3237,0.0,30.0,360.0,1.0,Urban
435,Female,NaN,0,Graduate,No,10047,0.0,NaN,240.0,1.0,Semiurban


In [15]:
x_test

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
350,Male,Yes,0,Graduate,No,9083,0.0,228.0,360.0,1.0,Semiurban
377,Male,Yes,0,Graduate,No,4310,0.0,130.0,360.0,NaN,Semiurban
163,Male,Yes,2,Graduate,No,4167,1447.0,158.0,360.0,1.0,Rural
609,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural
132,Male,No,0,Graduate,No,2718,0.0,70.0,360.0,1.0,Semiurban
...,...,...,...,...,...,...,...,...,...,...,...
231,Male,Yes,0,Graduate,NaN,3716,0.0,42.0,180.0,1.0,Rural
312,Female,No,0,Graduate,No,2507,0.0,56.0,360.0,1.0,Rural
248,Male,Yes,1,Graduate,No,2882,1843.0,123.0,480.0,1.0,Semiurban
11,Male,Yes,2,Graduate,NaN,2500,1840.0,109.0,360.0,1.0,Urban


In [19]:
# Transform Data
x_train_tran = preprocessor.fit_transform(x_train)
x_test_tran = preprocessor.transform(x_test)

In [20]:
preprocessor

,transformers,"[('ohe', ...), ('scaling', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None


In [17]:
x_train_tran

array([[ 0.,  1.,  0., ...,  1., 30.,  1.],
       [ 0.,  1.,  0., ...,  1., 30.,  1.],
       [ 0.,  1.,  0., ...,  1., 30.,  1.],
       ...,
       [ 0.,  0.,  1., ...,  1., 30.,  1.],
       [ 0.,  1.,  0., ...,  1., 20.,  1.],
       [ 0.,  0.,  1., ...,  1., 30.,  1.]], shape=(491, 16))

In [18]:
x_test_tran

array([[ 0.,  1.,  0., ...,  1., 30.,  1.],
       [ 0.,  1.,  0., ...,  1., 30.,  1.],
       [ 1.,  0.,  0., ...,  1., 30.,  1.],
       ...,
       [ 0.,  1.,  0., ...,  1., 40.,  1.],
       [ 0.,  0.,  1., ...,  1., 30.,  1.],
       [ 0.,  0.,  1., ...,  1., 15.,  1.]], shape=(123, 16))

In [19]:
y_test

350    Y
377    Y
163    Y
609    Y
132    Y
      ..
231    Y
312    Y
248    Y
11     Y
333    Y
Name: Loan_Status (Approved), Length: 123, dtype: object

In [20]:
y_train

83     N
90     Y
227    Y
482    Y
464    N
      ..
71     Y
106    Y
270    Y
435    Y
102    Y
Name: Loan_Status (Approved), Length: 491, dtype: object

In [21]:
# Encode Target
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [22]:
y_train

array([0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1,
       1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0,
       1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0,
       1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1,
       0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1,
       0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1,

In [23]:
y_test

array([1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1])

In [24]:
pickle.dump(preprocessor, open("preprocessor.pkl", "wb"))